# Oráculo normativo do TJPI no Google Colab

**Versão beta**, em fase de consolidação.

Converse com as normas do Judiciário do Piauí usando uma **IA gratuita que roda aqui mesmo, no computador do Google**, sem ChatGPT nem Gemini. Esta versão usa um banco vetorial de normas (RAG), reclassificação dos trechos e memória da conversa.

**Como usar**

1. No menu, abra **Ambiente de execução › Alterar o tipo de ambiente de execução**, escolha **GPU T4** e salve.
2. Clique no botão ▶ da etapa 1 e aguarde. Se aparecer um aviso sobre o notebook não ser do Google, escolha **Executar assim mesmo**.
3. Clique no ▶ da etapa 2. A página de conversa abre logo abaixo: é só perguntar.

**Antes de usar**

- **Tamanho:** a preparação baixa cerca de 11 GB, no computador do Google, não no seu.
- **Tempo:** a primeira resposta leva cerca de 6 minutos, porque a IA é carregada na memória, e as seguintes, cerca de 2 minutos. A preparação inicial leva cerca de 5 minutos. A demora vem dos limites computacionais do ambiente gratuito, não é um erro.
- **Atualização:** a base do notebook é atualizada mensalmente, e a do site, semanalmente, devido ao tempo necessário para ajustar cada base de dados.
- **Sessão temporária:** a versão gratuita do Colab tem limite de uso e pode recusar a GPU por algumas horas. Ao expirar, rode as etapas de novo.
- **A IA pode errar.** Confira sempre os artigos citados: o texto oficial prevalece.
- **Não digite dados de processos ou de pessoas.**

Iniciativa independente, sem vínculo oficial com o Tribunal. © 2026 Carlos Mendes de Sousa. Todos os direitos reservados, exceto quanto aos textos das normas, que são atos oficiais públicos.

In [ ]:
#@title 1. Preparar (uma vez por sessão)
MODELO = "gemma4:12b"  #@param ["gemma4:12b", "gemma4:e4b", "gemma4:e2b"]
import os, shutil, subprocess, sys, time, urllib.request

if not shutil.which("nvidia-smi"):
    raise SystemExit("GPU não encontrada. Abra Ambiente de execução › Alterar o tipo de ambiente de execução, "
                     "escolha GPU T4, salve e rode esta etapa de novo.")

print("Instalando programas…")
subprocess.run("apt-get -qq install -y zstd > /dev/null", shell=True, check=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1", shell=True, check=True)
# O chromadb traz o onnxruntime comum, que esconde a GPU do reclassificador: por isso ele sai depois.
subprocess.run("pip -q install tokenizers huggingface_hub chromadb gradio; pip -q uninstall -y onnxruntime onnxruntime-gpu > /dev/null 2>&1; "
               "pip -q install onnxruntime-gpu", shell=True, check=True)

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434", timeout=2)
        break
    except Exception:
        time.sleep(1)

for m in (MODELO, "bge-m3"):
    print(f"Baixando {m}…")
    subprocess.run(["ollama", "pull", m], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

SITE = "https://carlos-mendes-maestria.github.io/oraculo-normativo/rag/"
os.makedirs("/content/rag", exist_ok=True)
for nome in ("oraculo_rag.py", "interface.py"):
    urllib.request.urlretrieve(SITE + nome, "/content/rag/" + nome)
print("Pronto. Siga para a etapa 2.")

In [ ]:
#@title 2. Abrir o Oráculo
import sys
sys.path.insert(0, "/content/rag")
import interface, oraculo_rag

oraculo = oraculo_rag.Oraculo("/content/dados_oraculo", modelo=MODELO)
aviso = "a primeira resposta leva cerca de 6 minutos, porque a IA é carregada na memória, e as seguintes, cerca de 2 minutos. A preparação inicial leva cerca de 5 minutos. A demora vem dos limites computacionais do ambiente gratuito, não é um erro."
print("Pronto. A página de conversa abre abaixo.")
interface.criar(oraculo, aviso).queue().launch(share=False, height=900, inline=True, debug=False, quiet=True)